# ViT-Based Single Object Tracking (SOT) — OSTrack Tutorial

## What you'll learn
1. **What is SOT** and why it's challenging
2. **Evolution**: SiamFC → SiamRPN → OSTrack (ViT-based)
3. **One-stream design**: why processing template + search jointly works so well
4. **ViT anatomy**: patch embeddings, positional encoding, attention
5. **GOT-10k dataset**: structure, loading, and cropping strategy
6. **Model forward pass**: step-by-step with shapes
7. **Loss functions**: L1, GIoU, Focal — why each matters
8. **Training tips**: LR decay, mixed precision, gradient clipping
9. **Evaluation**: AO, SR50, SR75 metrics explained
10. **Visualization**: attention maps, tracking results

---
**Prerequisites**: PyTorch, timm, matplotlib, PIL

```bash
pip install torch torchvision timm matplotlib pillow tqdm pyyaml
```

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')  # Add project root

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Part 1: What is Single Object Tracking (SOT)?

**Task definition**: Given the first frame of a video with a target object annotated by a bounding box, localize that object in all subsequent frames.

```
Frame 1:  [Image] + [bbox: x1,y1,x2,y2]  ← initialization
Frame 2:  [Image]                          → predict [bbox]
Frame 3:  [Image]                          → predict [bbox]
...       ...
```

**Why is it hard?**
- Object appearance changes (viewpoint, lighting, deformation)
- Fast motion and motion blur
- Occlusion (object partially or fully hidden)
- Background clutter (similar-looking distractors)
- Scale changes

**Key constraint**: In test-time generalization benchmarks like GOT-10k, you train ONLY on the GOT-10k training set. The model must generalize to unseen object classes.

## Part 2: Evolution of Deep SOT Methods

### Era 1: Siamese Networks (2016–2019)

**SiamFC** (Bertinetto et al., 2016) — first end-to-end deep tracker:
- Template and search are processed by the **same** CNN backbone (Siamese = shared weights)
- Cross-correlation produces a response map: high score = object location

```
Template → CNN ─┐
                 ✕ cross-correlate → response map → peak = location
Search   → CNN ─┘
```

**SiamRPN** (Li et al., 2018):
- Added Region Proposal Network for joint classification + regression
- Much better accuracy, first tracker to run at >100 FPS

**SiamBAN, SiamCAR** (2020): Anchor-free versions, cleaner formulation

### Era 2: Transformer + Two-Stream (2021)

**TransT** (Chen et al., 2021):
- Replaces cross-correlation with transformer cross-attention
- Template and search still processed separately (CNN), then fused by attention

**STARK** (Yan et al., 2021):
- Full transformer encoder on concatenated template + search features
- But features come from a separate CNN — transformer is just the fusion module

### Era 3: One-Stream ViT (2022–present)

**OSTrack** (Ye et al., 2022) — the model we implement:
- **No separate CNN**: ViT processes raw patches from BOTH template and search
- Template and search tokens are concatenated before the FIRST transformer layer
- All 12 ViT layers perform cross-attention implicitly
- Simple MLP/Conv head for final prediction

```
Template patches ─┐
                   concat → [z₁...z₆₄ | x₁...x₂₅₆] → ViT L1 → ... → ViT L12 → Head → bbox
Search patches   ─┘
```

**Why is one-stream better?**
- Earlier, deeper interaction between template and search
- Pretrained ViT features are highly expressive
- Simpler architecture (no separate cross-attention module needed)
- Fewer parameters for similar performance

## Part 3: ViT Anatomy for Tracking

### 3.1 Patch Embedding

ViT treats an image as a sequence of patches. For tracking:
- Template: 128×128 image → (128/16)² = **64 patches**
- Search: 256×256 image → (256/16)² = **256 patches**
- Each patch: 16×16×3 = 768 values → projected to D-dim embedding

### 3.2 Positional Encoding

Since attention is permutation-invariant, we add position info:
- **Learnable** positional embeddings (separate for z and x)
- Initialized with 2D sinusoidal encoding for better inductive bias

In [ ]:
# Visualize 2D sinusoidal positional encoding
import sys
sys.path.insert(0, '..')
from lib.models.vit_backbone import get_2d_sincos_pos_embed

# Generate pos embed for 16x16 grid (search region)
embed_dim = 64  # Small dim for visualization
pos_embed = get_2d_sincos_pos_embed(embed_dim, 16, 16)  # (256, 64)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('2D Sinusoidal Positional Encoding Channels (16×16 grid)', fontsize=13)

for i, ax in enumerate(axes.flatten()):
    channel = pos_embed[:, i*8].reshape(16, 16)
    im = ax.imshow(channel, cmap='RdBu', aspect='equal')
    ax.set_title(f'Dim {i*8}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()
print('\nEach dimension encodes a different spatial frequency.')
print('Low-dim channels: coarse structure | High-dim: fine structure')

### 3.3 Self-Attention in Tracking

In OSTrack, after concatenating z and x tokens, each token can attend to every other token:

- **Search → Template attention**: "Does this search patch look like the template?"
- **Search → Search attention**: "Is this a distinctive/discriminative region?"
- **Template → Search attention**: "Where in the search region is the object?"

This is essentially cross-attention for free — no separate module needed!

In [ ]:
# Build a small OSTrack model and inspect it
from lib.models import build_ostrack_small

# Build without pretrained weights (for demo — no internet needed)
model = build_ostrack_small(pretrained=False)
model.eval()

print('OSTrack Architecture:')
print('=' * 50)
print(f'Template size: {model.backbone.template_size}×{model.backbone.template_size}')
print(f'Search size:   {model.backbone.search_size}×{model.backbone.search_size}')
print(f'Patch size:    {model.backbone.patch_size}×{model.backbone.patch_size}')
print(f'Embed dim:     {model.backbone.embed_dim}')
print(f'Num z tokens:  {model.backbone.num_z_patches}')
print(f'Num x tokens:  {model.backbone.num_x_patches}')
print(f'Total tokens:  {model.backbone.num_z_patches + model.backbone.num_x_patches}')

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## Part 4: Forward Pass — Step by Step

Let's trace a forward pass through OSTrack with real tensors.

In [ ]:
# Create dummy inputs (batch of 2)
B = 2
template = torch.randn(B, 3, 128, 128)  # (B, C, Hz, Wz)
search   = torch.randn(B, 3, 256, 256)  # (B, C, Hx, Wx)

print('Input shapes:')
print(f'  template: {tuple(template.shape)}')
print(f'  search:   {tuple(search.shape)}')

# Step 1: Patch embedding
backbone = model.backbone
z_tokens = backbone.patch_embed_image(template)  # (B, 64, 384)
x_tokens = backbone.patch_embed_image(search)    # (B, 256, 384)

print('\nAfter patch embedding:')
print(f'  z_tokens: {tuple(z_tokens.shape)}  [B, Nz, D]')
print(f'  x_tokens: {tuple(x_tokens.shape)}  [B, Nx, D]')

# Step 2: Add positional embeddings
z_tokens = z_tokens + backbone.pos_embed_z
x_tokens = x_tokens + backbone.pos_embed_x
print('\nAfter positional embedding: (same shapes, pos info added)')

# Step 3: Concatenate
tokens = torch.cat([z_tokens, x_tokens], dim=1)  # (B, 320, 384)
print(f'\nAfter concatenation: {tuple(tokens.shape)}  [B, Nz+Nx, D]')
print(f'  = {backbone.num_z_patches} z + {backbone.num_x_patches} x = {tokens.shape[1]} total tokens')

# Step 4: Transformer blocks (all 12 layers)
with torch.no_grad():
    tokens_out = backbone.blocks(tokens)  # (B, 320, 384)
    tokens_out = backbone.norm(tokens_out)

print(f'\nAfter ViT transformer: {tuple(tokens_out.shape)}')

# Step 5: Split back
z_out = tokens_out[:, :backbone.num_z_patches]  # (B, 64, 384)
x_out = tokens_out[:, backbone.num_z_patches:]  # (B, 256, 384)

print(f'\nSplit output:')
print(f'  z_out: {tuple(z_out.shape)}  (template tokens, enriched by x interaction)')
print(f'  x_out: {tuple(x_out.shape)}  (search tokens, enriched by z interaction)')

# Step 6: Head prediction from search tokens
with torch.no_grad():
    predictions = model.head(x_out)

print(f'\nPredicted boxes: {tuple(predictions["pred_boxes"].shape)}')
print(f'  Values: {predictions["pred_boxes"][0].numpy()}')
print(f'  Format: [x1, y1, x2, y2] normalized to [0,1]')

In [ ]:
# Full forward pass (cleaner)
with torch.no_grad():
    output = model(template, search)

print('Full forward pass output:')
for k, v in output.items():
    if v is not None:
        print(f'  {k}: {tuple(v.shape) if hasattr(v, "shape") else v}')

## Part 5: GOT-10k Dataset and Crop Strategy

### 5.1 Dataset Structure

```
got10k/train/
├── GOT-10k_Train_000001/
│   ├── 00000001.jpg  ... 00000180.jpg   ← video frames
│   ├── groundtruth.txt                  ← x,y,w,h per line
│   └── meta_info.ini                    ← object class, attributes
├── GOT-10k_Train_000002/
...
```

### 5.2 Crop Strategy (SiamFC-style)

The key insight: we don't feed the whole frame to the model. We crop a fixed-size region centered on the object.

The crop size formula:
```
context = (w + h) / 2 * factor
crop_size = sqrt((w + context) * (h + context))
```

This produces a crop whose size scales with the object but includes surrounding context.

In [ ]:
# Demonstrate the crop strategy on a synthetic example
import math

def visualize_crop_strategy():
    # Create a synthetic 480x640 "image"
    img = np.ones((480, 640, 3), dtype=np.uint8) * 200
    
    # Random background texture
    np.random.seed(42)
    img += np.random.randint(-20, 20, img.shape, dtype=np.int8).astype(np.uint8)
    
    # Simulate an object at center
    obj_x1, obj_y1, obj_x2, obj_y2 = 250, 180, 390, 300
    img[obj_y1:obj_y2, obj_x1:obj_x2] = [100, 150, 200]
    
    bbox = np.array([obj_x1, obj_y1, obj_x2, obj_y2], dtype=np.float32)
    cx, cy = (bbox[0]+bbox[2])/2, (bbox[1]+bbox[3])/2
    w, h = bbox[2]-bbox[0], bbox[3]-bbox[1]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Crop Strategy: Template (2× context) vs Search (4× context)', fontsize=13)
    
    # Original frame with object
    ax = axes[0]
    ax.imshow(img)
    ax.add_patch(patches.Rectangle(
        (obj_x1, obj_y1), w, h,
        linewidth=2, edgecolor='red', facecolor='none', label='Object bbox'
    ))
    ax.set_title('Original Frame')
    ax.legend()
    
    # Template crop (2× context)
    for ax, factor, label, crop_size in [
        (axes[1], 2.0, 'Template (factor=2.0)', 128),
        (axes[2], 4.0, 'Search (factor=4.0)', 256),
    ]:
        context = (w + h) / 2 * factor
        csz = math.sqrt((w + context) * (h + context))
        cx1 = max(0, int(cx - csz/2))
        cy1 = max(0, int(cy - csz/2))
        cx2 = min(640, int(cx + csz/2))
        cy2 = min(480, int(cy + csz/2))
        
        crop = img[cy1:cy2, cx1:cx2]
        from PIL import Image
        crop_pil = Image.fromarray(crop).resize((crop_size, crop_size))
        
        ax.imshow(np.array(crop_pil))
        scale = crop_size / (cx2 - cx1)
        bx1 = (obj_x1 - cx1) * scale
        by1 = (obj_y1 - cy1) * scale
        bw = w * scale
        bh = h * scale
        ax.add_patch(patches.Rectangle(
            (bx1, by1), bw, bh,
            linewidth=2, edgecolor='lime', facecolor='none'
        ))
        ax.set_title(f'{label}\ncrop_sz={csz:.0f}px → resized to {crop_size}px')
    
    plt.tight_layout()
    plt.show()

visualize_crop_strategy()
print('\nKey insight: The object is always centered in the crop.')
print('Template (factor=2): tight crop, mostly object')
print('Search (factor=4): wider crop, includes surrounding context for motion')

## Part 6: Loss Functions

OSTrack uses three complementary losses:

| Loss | Why? |
|------|------|
| **L1** | Fast, stable coordinate regression |
| **GIoU** | Scale-invariant, handles non-overlapping boxes gracefully |
| **Focal** | Score map supervision, handles class imbalance |

**Why not just use L1?** L1 treats all coordinates equally. A 10px error on a 20px box is much worse than on a 200px box. GIoU is scale-invariant.

**Why GIoU over IoU?** IoU gradient is 0 when boxes don't overlap. GIoU adds a penalty term for the enclosing box area, providing gradient even for non-overlapping predictions.

In [ ]:
# Visualize GIoU vs IoU behavior
from lib.models.head import giou_loss_fn

# Compare loss landscapes: vary predicted box x1 while keeping x2 fixed
gt_box = torch.tensor([[0.3, 0.3, 0.7, 0.7]])  # center box

x1_vals = np.linspace(0.0, 0.9, 100)
iou_losses, giou_losses, l1_losses = [], [], []

from lib.utils.box_ops import box_iou

for x1 in x1_vals:
    pred = torch.tensor([[x1, 0.3, min(x1+0.3, 1.0), 0.7]])
    
    # IoU loss (manually)
    iou = box_iou(pred, gt_box)[0, 0].item()
    iou_losses.append(1 - iou)
    
    # GIoU loss
    giou_loss = giou_loss_fn(pred, gt_box).item()
    giou_losses.append(giou_loss)
    
    # L1 loss
    l1 = torch.nn.functional.l1_loss(pred, gt_box).item()
    l1_losses.append(l1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x1_vals, iou_losses, 'b-', label='1 - IoU', linewidth=2)
ax.plot(x1_vals, giou_losses, 'r-', label='GIoU Loss', linewidth=2)
ax.plot(x1_vals, [v/max(l1_losses) for v in l1_losses], 'g--', label='L1 (normalized)', linewidth=2)
ax.axvspan(0.0, 0.3, alpha=0.1, color='gray', label='No overlap region')
ax.axvline(x=0.3, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel('Predicted x1 (x2 = x1 + 0.3)', fontsize=11)
ax.set_ylabel('Loss value', fontsize=11)
ax.set_title('Loss Landscape: IoU vs GIoU vs L1\n(GT box: [0.3, 0.3, 0.7, 0.7])', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Key observations:')
print('  • IoU loss = 0 gradient when boxes dont overlap (x1 < 0.3 region)')
print('  • GIoU provides gradient everywhere → better for training')
print('  • L1 is linear → fast convergence early in training')
print('  → We use ALL THREE: L = 5·L1 + 2·GIoU + 1·Focal')

## Part 7: Training Strategy

### 7.1 Layer-wise Learning Rate Decay

When fine-tuning a large pretrained ViT, use **different learning rates** for backbone vs head:

```
Backbone LR = base_lr × 0.1 = 4e-5    (gentle fine-tuning)
Head LR     = base_lr       = 4e-4    (train from scratch)
```

Why? The backbone is pretrained on ImageNet — it already has good features. We don't want to destroy them with a large LR. The head is randomly initialized and needs to learn from scratch.

In [ ]:
# Visualize training schedule
import math

num_epochs = 300
warmup_epochs = 2
base_lr = 4e-4
backbone_lr_factor = 0.1

def cosine_schedule(epoch):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs)
    return 0.5 * (1 + math.cos(math.pi * progress))

epochs = list(range(num_epochs))
head_lrs = [base_lr * cosine_schedule(e) for e in epochs]
backbone_lrs = [lr * backbone_lr_factor for lr in head_lrs]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(epochs, head_lrs, 'b-', label='Head LR', linewidth=2)
ax.plot(epochs, backbone_lrs, 'r-', label='Backbone LR (×0.1)', linewidth=2)
ax.axvspan(0, warmup_epochs, alpha=0.2, color='orange', label='Linear warmup')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Learning Rate', fontsize=11)
ax.set_title('Cosine LR Schedule with Linear Warmup (300 epochs)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0e}'))
plt.tight_layout()
plt.show()

print('Training tips:')
print('  • Warmup prevents instability at the start of training')
print('  • Cosine decay: smooth reduction, avoids aggressive drops')
print('  • AdamW (not Adam): weight decay applied to weights, NOT biases/norms')
print('  • Gradient clipping (max_norm=0.1): prevents transformer gradient explosions')
print('  • Mixed precision (AMP): ~2× speedup on modern GPUs')

### 7.2 Sample Training Step

In [ ]:
from lib.models.head import TrackingLoss
import torch.optim as optim

# Create a small model for demo
model_train = build_ostrack_small(pretrained=False)
model_train.train()

# Optimizer with separate LRs
param_groups = [
    {'params': list(model_train.backbone.parameters()), 'lr': 4e-5, 'name': 'backbone'},
    {'params': list(model_train.head.parameters()), 'lr': 4e-4, 'name': 'head'},
]
optimizer = optim.AdamW(param_groups, weight_decay=1e-4)
criterion = TrackingLoss(l1_weight=5.0, giou_weight=2.0, focal_weight=1.0)

# Dummy batch
B = 4
template = torch.randn(B, 3, 128, 128)
search   = torch.randn(B, 3, 256, 256)
gt_boxes = torch.rand(B, 4)  # Random normalized boxes
# Ensure x2 > x1, y2 > y1
gt_boxes[:, 2] = gt_boxes[:, 0] + 0.2
gt_boxes[:, 3] = gt_boxes[:, 1] + 0.2
gt_boxes = gt_boxes.clamp(0, 1)

# Forward pass
optimizer.zero_grad()
z_tokens, x_tokens = model_train.backbone(template, search)
predictions = model_train.head(x_tokens)
losses = criterion(predictions['pred_boxes'], gt_boxes)

# Backward
losses['total'].backward()
torch.nn.utils.clip_grad_norm_(model_train.parameters(), max_norm=0.1)
optimizer.step()

print('Training step complete!')
print(f"  Total loss:  {losses['total'].item():.4f}")
print(f"  L1 loss:     {losses['l1'].item():.4f}")
print(f"  GIoU loss:   {losses['giou'].item():.4f}")

print(f"\n  Predicted boxes (first 2):")
print(f"    {predictions['pred_boxes'][0].detach().numpy().round(3)}")
print(f"    {predictions['pred_boxes'][1].detach().numpy().round(3)}")
print(f"  Ground truth boxes (first 2):")
print(f"    {gt_boxes[0].numpy().round(3)}")
print(f"    {gt_boxes[1].numpy().round(3)}")

## Part 8: Evaluation Metrics

GOT-10k uses three primary metrics:

| Metric | Formula | Meaning |
|--------|---------|--------|
| **AO** | mean(IoU) | Average Overlap — primary GOT-10k metric |
| **SR₀.₅** | % frames with IoU ≥ 0.5 | Success Rate (loose threshold) |
| **SR₀.₇₅** | % frames with IoU ≥ 0.75 | Success Rate (strict threshold) |

**State-of-the-art numbers on GOT-10k test** (from papers):
```
Method          AO     SR50   SR75
SiamFC         0.374  0.404  0.144
SiamRPN++      0.518  0.616  0.325
TransT         0.671  0.762  0.576
OSTrack-384    0.739  0.836  0.654   ← what we're building
```

In [ ]:
from lib.utils.metrics import evaluate_sequence, compute_iou
import numpy as np

# Simulate a tracking sequence result
np.random.seed(7)
n_frames = 100

# Simulate GT boxes (slowly moving object)
gt_boxes = []
x, y, w, h = 100, 100, 60, 40
for i in range(n_frames):
    x += np.random.randn() * 2
    y += np.random.randn() * 1.5
    gt_boxes.append(np.array([x, y, x+w, y+h]))

# Simulate tracker predictions (good tracker with some error)
pred_boxes = []
for gt in gt_boxes:
    noise = np.random.randn(4) * 8  # ~8px noise
    pred = gt + noise
    pred_boxes.append(pred)

# Evaluate
results = evaluate_sequence(pred_boxes, gt_boxes)

print('Sequence Evaluation Results:')
print(f'  AO   (Average Overlap):      {results["AO"]:.3f}')
print(f'  SR50 (% frames, IoU ≥ 0.5):  {results["SR50"]:.3f}')
print(f'  SR75 (% frames, IoU ≥ 0.75): {results["SR75"]:.3f}')
print(f'  AUC  (success curve area):   {results["AUC"]:.3f}')
print(f'  Prec (% frames, err ≤ 20px): {results["Precision@20"]:.3f}')

# Plot frame-by-frame IoU
ious = [compute_iou(p, g) for p, g in zip(pred_boxes, gt_boxes)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Per-frame IoU
axes[0].plot(ious, 'b-', alpha=0.7, linewidth=1)
axes[0].axhline(y=0.5, color='r', linestyle='--', label='SR50 threshold')
axes[0].axhline(y=0.75, color='g', linestyle='--', label='SR75 threshold')
axes[0].fill_between(range(n_frames), ious, alpha=0.2)
axes[0].set_xlabel('Frame', fontsize=11)
axes[0].set_ylabel('IoU', fontsize=11)
axes[0].set_title(f'Per-frame IoU | AO={results["AO"]:.3f}', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1)

# Success curve
thresholds = np.linspace(0, 1, 101)
success_rates = [(np.array(ious) >= t).mean() for t in thresholds]
axes[1].plot(thresholds, success_rates, 'b-', linewidth=2, label=f'AUC={results["AUC"]:.3f}')
axes[1].fill_between(thresholds, success_rates, alpha=0.2)
axes[1].set_xlabel('Overlap threshold', fontsize=11)
axes[1].set_ylabel('Success rate', fontsize=11)
axes[1].set_title('Success Curve', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Part 9: Understanding Attention in Tracking

One of the biggest advantages of ViT-based trackers: we can **visualize attention maps** to understand what the model looks at.

In OSTrack, at each layer the model computes:
- Search→Template attention: which search patches are most similar to the template?
- Search self-attention: which search patches are most distinctive?

These attention patterns are what enables the model to localize the object.

In [ ]:
# Analyze attention patterns with dummy data
# (With a pretrained model, these patterns are meaningful)

model_eval = build_ostrack_small(pretrained=False)
model_eval.eval()

# Create inputs with a "signal" region
template = torch.zeros(1, 3, 128, 128)
search = torch.zeros(1, 3, 256, 256)

# Put a bright region in the template and in the search (object location)
template[:, :, 32:96, 32:96] = 1.0   # Object in template center
search[:, :, 80:160, 96:176] = 1.0   # Object in search region (shifted)

# Hook to capture attention weights
attention_maps = []

def attn_hook(module, input, output):
    # For nn.MultiheadAttention, output is (attn_output, attn_weights)
    # This hook captures the attention output
    pass  # Simplified — real implementation would capture weights

with torch.no_grad():
    z_tokens, x_tokens = model_eval.backbone(template, search)

# Visualize the search token activation patterns
Nz = model_eval.backbone.num_z_patches
Hx = Wx = model_eval.backbone.search_size // model_eval.backbone.patch_size

# x_tokens has shape (1, 256, 384) — visualize norm as activation map
token_norms = x_tokens[0].norm(dim=-1).detach().numpy()  # (256,)
activation_map = token_norms.reshape(Hx, Wx)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Search Token Activations after ViT', fontsize=13)

axes[0].imshow(template[0].permute(1,2,0).clamp(0,1).numpy())
axes[0].set_title('Template (128×128)')
axes[0].axis('off')

axes[1].imshow(search[0].permute(1,2,0).clamp(0,1).numpy())
axes[1].set_title('Search (256×256)')
axes[1].axis('off')

im = axes[2].imshow(activation_map, cmap='hot')
axes[2].set_title('Search Token Norms\n(proxy for attention)')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

print('With pretrained weights, the hot regions would correspond to object-like features.')
print('The pretrained ViT has learned to attend to semantically meaningful regions.')

## Part 10: End-to-End Tracking Demo

Let's simulate the online tracking loop with a simple synthetic sequence.

In [ ]:
# Simulate online tracking (synthetic sequence)
from lib.datasets.got10k import crop_and_resize
import math

def create_synthetic_sequence(n_frames=20, im_size=480):
    """Create a synthetic sequence with a moving colored object."""
    frames = []
    boxes = []
    
    # Object starts at center, moves right and down
    x, y = im_size // 4, im_size // 3
    w, h = 60, 45
    
    np.random.seed(0)
    for i in range(n_frames):
        # Background
        frame = np.random.randint(80, 120, (im_size, im_size, 3), dtype=np.uint8)
        # Add some texture
        for _ in range(20):
            rx, ry = np.random.randint(0, im_size, 2)
            rw, rh = np.random.randint(10, 40, 2)
            frame[ry:ry+rh, rx:rx+rw] = np.random.randint(60, 160, 3)
        
        # Object (bright blue box)
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(im_size, x + w), min(im_size, y + h)
        frame[y1:y2, x1:x2] = [50, 100, 220]
        
        frames.append(Image.fromarray(frame))
        boxes.append(np.array([x1, y1, x2, y2], dtype=np.float32))
        
        # Move object
        x += 5 + np.random.randn() * 2
        y += 3 + np.random.randn() * 1
    
    return frames, boxes

frames, gt_boxes = create_synthetic_sequence(n_frames=15)

# Show a few frames
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for i, ax in enumerate(axes):
    frame_np = np.array(frames[i*3])
    box = gt_boxes[i*3]
    frame_np = frame_np.copy()
    x1, y1, x2, y2 = [int(v) for v in box]
    frame_np[y1:y1+3, x1:x2] = [255, 0, 0]
    frame_np[y2-3:y2, x1:x2] = [255, 0, 0]
    frame_np[y1:y2, x1:x1+3] = [255, 0, 0]
    frame_np[y1:y2, x2-3:x2] = [255, 0, 0]
    ax.imshow(frame_np)
    ax.set_title(f'Frame {i*3}')
    ax.axis('off')

plt.suptitle('Synthetic Tracking Sequence (red box = ground truth)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Simulate the tracking loop (without pretrained weights — just architecture demo)
from lib.tracking import OSTracker

# Note: Without pretrained weights, predictions are random.
# This demonstrates the INTERFACE and data flow.
# With a trained model (python train.py), predictions become meaningful.

tracker = OSTracker(
    model=build_ostrack_small(pretrained=False),
    device='cpu',
    template_size=128,
    search_size=256,
)

# Initialize with first frame
init_box = gt_boxes[0]  # [x1, y1, x2, y2]
tracker.init(frames[0], init_box)
print(f'Tracker initialized with box: {init_box.astype(int)}')

# Track subsequent frames
pred_boxes = [init_box]
for i, frame in enumerate(frames[1:], 1):
    pred = tracker.track(frame)
    pred_boxes.append(pred)
    iou = compute_iou(pred, gt_boxes[i])
    print(f'Frame {i:2d}: pred={pred.astype(int)}, gt={gt_boxes[i].astype(int)}, IoU={iou:.3f}')

print('\nNote: IoU is near-zero without pretrained weights.')
print('After training (python train.py), the model will predict accurately.')

## Summary & Next Steps

### What we built

A complete OSTrack implementation:
- **ViT backbone** (`lib/models/vit_backbone.py`): One-stream processing of template + search
- **Corner head** (`lib/models/head.py`): Predicts bounding box from search tokens
- **GOT-10k dataset** (`lib/datasets/got10k.py`): Loading, cropping, augmentation
- **Online tracker** (`lib/tracking/tracker.py`): Stateful tracker for video
- **Training script** (`train.py`): Full training loop with AMP, grad clip, cosine LR
- **Evaluation script** (`evaluate.py`): Computes AO, SR50, SR75 on GOT-10k

### Suggested experiments

1. **Ablation: head type** — compare `corner` vs `mlp` head
2. **Ablation: search factor** — try 3.0 vs 4.0 vs 5.0
3. **Backbone: ViT-Small vs ViT-Base** — performance vs speed tradeoff
4. **Template update** — try updating the template every N frames with the new prediction
5. **Multi-template** — store multiple templates (first frame + recent frame)

### Related papers to read

| Paper | Key contribution |
|-------|----------------|
| OSTrack (2022) | One-stream ViT tracking |
| MixFormer (2022) | Mixed attention module |
| AQATrack (2024) | Adaptive query tracking |
| SiamFC (2016) | First fully conv Siamese tracker |
| TransT (2021) | Transformer cross-attention for tracking |

### To train on GOT-10k

```bash
# 1. Download dataset
python scripts/download_got10k.py --info

# 2. Train (ViT-Small, ~2 days on single A100)
python train.py --config configs/ostrack_small.yaml

# 3. Evaluate on val set
python evaluate.py --checkpoint output/ostrack_small/best.pth --split val

# 4. Demo on your own video
python demo.py --checkpoint output/ostrack_small/best.pth --video my_video.mp4
```